<a href="https://colab.research.google.com/github/mitalidaduria/enterprise-data-platform/blob/main/Execution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!rm -rf enterprise-data-platform
!git clone https://github.com/mitalidaduria/enterprise-data-platform.git
import os
os.chdir("enterprise-data-platform")
print("📂 Current Directory:", os.getcwd())

Cloning into 'enterprise-data-platform'...
remote: Enumerating objects: 62, done.
remote: Counting objects: 100% (62/62), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 62 (delta 16), reused 8 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (62/62), 30.72 KiB | 3.84 MiB/s, done.
Resolving deltas: 100% (16/16), done.
📂 Current Directory: /content/enterprise-data-platform


In [2]:
!pip install pyspark -q

print("1️⃣ Generating raw synthetic data...")
!python pipelines/generate_data.py

print("2️⃣ Running PySpark ingestion & PII hashing...")
!python pipelines/ingest_pyspark.py

print("3️⃣ Executing Data Quality Gates & Quarantine...")
!python pipelines/data_quality.py

print("4️⃣ Running MDM Entity Resolution & Golden Record generation...")
!python pipelines/entity_resolution.py

print("\n🎉 All pipeline stages completed successfully!")

1️⃣ Generating raw synthetic data...
  File "/content/enterprise-data-platform/pipelines/generate_data.py", line 35
    raw_email = f"{first_name.lower()}.{last_name.lower()}{i}@"{domain}
                                                               ^
SyntaxError: invalid syntax
2️⃣ Running PySpark ingestion & PII hashing...
[0.034s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.034s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Using Spark's 

In [4]:
# 1. Overwrite pipelines/generate_data.py with clean, bug-free code
clean_generator_code = '''"""
Enterprise Data Platform: Raw Synthetic Data Generator
Generates mock multi-source operational datasets (Billing & Shipping)
with realistic PII, typos, and overlapping entity attributes for MDM testing.
"""

import csv
import hashlib
import os
import random
import uuid

DATA_DIR = os.path.join(os.path.dirname(__file__), "raw_data")
os.makedirs(DATA_DIR, exist_ok=True)

NUM_RECORD_PAIRS = 1000

FIRST_NAMES = ["Robert", "Bob", "Rob", "William", "Bill", "Elizabeth", "Liz", "Michael", "Mike", "Sarah"]
LAST_NAMES = ["Jones", "Smith", "Taylor", "Brown", "Wilson", "Davies", "Evans", "Thomas", "Johnson"]
STREETS = ["123 Main St", "456 Oak Ave", "789 Pine Rd", "101 Maple Dr", "202 Birch Ln"]
DOMAINS = ["gmail.com", "yahoo.com", "hotmail.com", "enterprise.org"]

def generate_datasets():
    billing_rows = []
    shipping_rows = []

    for i in range(NUM_RECORD_PAIRS):
        first_name = random.choice(FIRST_NAMES)
        last_name = random.choice(LAST_NAMES)
        full_name = f"{first_name} {last_name}"
        domain = random.choice(DOMAINS)

        raw_email = f"{first_name.lower()}.{last_name.lower()}{i}@{domain}"

        billing_id = f"BIL-{uuid.uuid4().hex[:8].upper()}"
        credit_card_hash = hashlib.sha256(f"CARD-{i}".encode()).hexdigest()
        billing_address = random.choice(STREETS)

        billing_rows.append([
            billing_id,
            full_name,
            raw_email,
            credit_card_hash,
            billing_address
        ])

        shipping_id = f"SHP-{uuid.uuid4().hex[:8].upper()}"
        recipient_name = f"{first_name[0]}. {last_name}"
        phone_number = f"555-{random.randint(100, 999)}-{random.randint(1000, 9999)}"
        street_address = billing_address

        shipping_rows.append([
            shipping_id,
            recipient_name,
            raw_email,
            street_address,
            phone_number
        ])

    billing_path = os.path.join(DATA_DIR, "raw_billing.csv")
    with open(billing_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["billing_id", "full_name", "email", "credit_card_hash", "billing_address"])
        writer.writerows(billing_rows)

    shipping_path = os.path.join(DATA_DIR, "raw_shipping.csv")
    with open(shipping_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["shipping_id", "recipient_name", "email", "street_address", "phone_number"])
        writer.writerows(shipping_rows)

    print(f"✅ Generated {NUM_RECORD_PAIRS} synthetic raw records successfully.")

if __name__ == "__main__":
    generate_datasets()
'''

with open("pipelines/generate_data.py", "w") as f:
    f.write(clean_generator_code)

print("✅ Fixed generate_data.py! Now running full pipeline...")

# 2. Run the full pipeline sequentially
import subprocess
subprocess.run(["python", "pipelines/generate_data.py"], check=True)
subprocess.run(["python", "pipelines/ingest_pyspark.py"], check=True)
subprocess.run(["python", "pipelines/data_quality.py"], check=True)
subprocess.run(["python", "pipelines/entity_resolution.py"], check=True)
subprocess.run(["python", "infrastructure/load_db.py"], check=True)

print("\n✨ Pipeline execution complete! Testing API Client...")

# 3. Test API Gateway
from fastapi.testclient import TestClient
from gateway.app import app

client = TestClient(app)
res = client.get("/customers?limit=3")
print("\n🔥 STATUS CODE:", res.status_code)
print("✨ DATA:", res.json())

✅ Fixed generate_data.py! Now running full pipeline...

✨ Pipeline execution complete! Testing API Client...

🔥 STATUS CODE: 200
✨ DATA: [{'golden_customer_id': '35508333febf1ddc32a8941d9a694274ec99dee44e334236f49ac2a2c9cf5607', 'primary_name': 'Sarah Evans', 'primary_email_hash': '004fccf9d4bdac7ecc346ff855a720a8b6dd2a7b0be7ff2911e4cca369538d10', 'primary_phone': '555-380-9240', 'primary_address': '202 Birch Ln', 'billing_linkage_id': 'BIL-3C09B869', 'shipping_linkage_id': 'SHP-A949B6E3', 'total_source_linkages': 2, 'pipeline_version': 'v1.0.0'}, {'golden_customer_id': '23c90512dedeff5f41785f3e9cc0166a0c6eb30f7470a989d9fd47edfefa1c99', 'primary_name': 'Elizabeth Wilson', 'primary_email_hash': '0080ecc0cac5e3c6cb0ae5e53d5cc3e0f6308b10baa6647d0b6103a8d260e873', 'primary_phone': '555-587-8219', 'primary_address': '123 Main St', 'billing_linkage_id': 'BIL-F0798E8C', 'shipping_linkage_id': 'SHP-9D27688A', 'total_source_linkages': 2, 'pipeline_version': 'v1.0.0'}, {'golden_customer_id': '3f

In [5]:
import sqlite3
import pandas as pd
from IPython.display import display

# Connect to your SQLite database and load the table into a Pandas DataFrame
conn = sqlite3.connect("infrastructure/enterprise_data.db")
df = pd.read_sql("SELECT * FROM dim_customer_golden", conn)
conn.close()

print(f"📊 Total Golden Records in Grid: {len(df)}")

# Display as an interactive, scrollable table grid in Colab
display(df)

📊 Total Golden Records in Grid: 1000


,golden_customer_id,primary_name,primary_email_hash,primary_phone,primary_address,billing_linkage_id,shipping_linkage_id,total_source_linkages,created_at,updated_at,pipeline_version
0,35508333febf1ddc32a8941d9a694274ec99dee44e3342...,Sarah Evans,004fccf9d4bdac7ecc346ff855a720a8b6dd2a7b0be7ff...,555-380-9240,202 Birch Ln,BIL-3C09B869,SHP-A949B6E3,2,2026-09-09 19:13:41.168098,2026-09-09 19:13:41.168098,v1.0.0
1,23c90512dedeff5f41785f3e9cc0166a0c6eb30f7470a9...,Elizabeth Wilson,0080ecc0cac5e3c6cb0ae5e53d5cc3e0f6308b10baa664...,555-587-8219,123 Main St,BIL-F0798E8C,SHP-9D27688A,2,2026-09-09 19:13:41.168098,2026-09-09 19:13:41.168098,v1.0.0
2,3f4ab1c5199a4601ceec1e426d4739df57cecd715f1ceb...,Sarah Jones,00a875eb5e06a83c03bee8c6b8c98660b319bc210feb0a...,555-666-4289,123 Main St,BIL-80345E7C,SHP-EAC5A077,2,2026-09-09 19:13:41.168098,2026-09-09 19:13:41.168098,v1.0.0
3,b8933615783d7857f3931cc807e59082292ae24634a494...,Bob Smith,00be02504a7ac524dfc217187454ce06a0ccd8b2697513...,555-842-4658,456 Oak Ave,BIL-EC1A332C,SHP-E30B56C4,2,2026-09-09 19:13:41.168098,2026-09-09 19:13:41.168098,v1.0.0
4,6564bf7efd4f7e2e4cc0f85ba73c22c59257b659e31959...,Mike Wilson,00fc558beb4af4e63671b02d8ef772a391fec83e898c64...,555-706-4009,202 Birch Ln,BIL-0B749636,SHP-08010F39,2,2026-09-09 19:13:41.168098,2026-09-09 19:13:41.168098,v1.0.0
...,...,...,...,...,...,...,...,...,...,...,...
995,6fee8e1b9d5e3445dcec32ecaf0c94aedbf55e3bfe6d53...,Bill Taylor,ff150be6da22a8c9b3fdefccc6619787962ba88ceecd10...,555-517-1274,202 Birch Ln,BIL-E26D30BA,SHP-93B8A706,2,2026-09-09 19:13:41.168098,2026-09-09 19:13:41.168098,v1.0.0
996,5baafec0c7670bbc985a5f9a234ae1e0df3fd0b359bc17...,Michael Smith,ff7f33189e4a8175cdaba59de5abe5b9d110eb94727c8d...,555-748-3603,456 Oak Ave,BIL-7438885F,SHP-9E688284,2,2026-09-09 19:13:41.168098,2026-09-09 19:13:41.168098,v1.0.0
997,1a2250b10ddbdadbe027154f292e89b9dcc2ed810f6bad...,Bill Davies,ffacee01afa77d577edc386b4caae3f83837cfa2c39cdd...,555-390-1819,456 Oak Ave,BIL-0A085C9C,SHP-2967EEA1,2,2026-09-09 19:13:41.168098,2026-09-09 19:13:41.168098,v1.0.0
998,915ccfa85784c27fae98bde4b3f20341c85e560064d836...,Robert Johnson,ffcc1698162d6942aec65c0279b24ede0c267c7116aa16...,555-193-2267,456 Oak Ave,BIL-7BDFA1CC,SHP-43E5D8DB,2,2026-09-09 19:13:41.168098,2026-09-09 19:13:41.168098,v1.0.0


**Scikit-Learn Random Forest Classifier**

In [6]:
# 1. Pull the latest files from GitHub (fetches models/train_model.py)
!git pull origin main

# 2. Execute the ML training and decision engine
print("🤖 Training Machine Learning Decision Model...")
!python models/train_model.py

remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 10 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (10/10), 10.79 KiB | 526.00 KiB/s, done.
From https://github.com/mitalidaduria/enterprise-data-platform
 * branch            main       -> FETCH_HEAD
   5952444..1316fdd  main       -> origin/main
Updating 5952444..1316fdd
Fast-forward
 Execution.ipynb       | 717 ++++++++++++++++++++++++++++++++++++++++++++++++++
 models/train_model.py |  82 ++++++
 2 files changed, 799 insertions(+)
 create mode 100644 Execution.ipynb
 create mode 100644 models/train_model.py
🤖 Training Machine Learning Decision Model...
📊 Loaded 1000 Golden Records for Feature Engineering.

🎯 --- Model Training & Evaluation Metrics ---
  * Accuracy : 0.9850
  * Precision: 0.9850
  * Recall   : 1.0000
  * F1-Score : 0.9924

💾 Serialized Model saved successfully to '/content/enterprise-data-plat

In [7]:
# 1. Pull your updated Day 10 app.py from GitHub
!git pull origin main

# 2. Reload the module cache and test the real-time ML decision endpoint
import sys
import importlib
import sqlite3

if 'gateway.app' in sys.modules:
    importlib.reload(sys.modules['gateway.app'])

from fastapi.testclient import TestClient
from gateway.app import app

client = TestClient(app)

# Fetch a sample golden_customer_id from your SQLite database
conn = sqlite3.connect("infrastructure/enterprise_data.db")
cursor = conn.cursor()
cursor.execute("SELECT golden_customer_id FROM dim_customer_golden LIMIT 1")
sample_id = cursor.fetchone()[0]
conn.close()

print(f"🔍 Testing live ML prediction for Customer ID: {sample_id[:12]}...")

# Hit the new /predict/{customer_id} endpoint
response = client.get(f"/predict/{sample_id}")
print("\n🔥 PREDICTION STATUS CODE:", response.status_code)
print("✨ PREDICTION RESPONSE JSON:", response.json())

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 4 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 2.92 KiB | 2.92 MiB/s, done.
From https://github.com/mitalidaduria/enterprise-data-platform
 * branch            main       -> FETCH_HEAD
   1316fdd..679e922  main       -> origin/main
Updating 1316fdd..679e922
Fast-forward
 gateway/app.py | 152 +++++++++++++++++++++++++++++++++++++++++++++------------
 1 file changed, 121 insertions(+), 31 deletions(-)
🔍 Testing live ML prediction for Customer ID: 35508333febf...

🔥 PREDICTION STATUS CODE: 500
✨ PREDICTION RESPONSE JSON: {'detail': 'ML Model is not loaded. Train model first.'}


In [9]:
import sys
import importlib
import sqlite3

# 1. Ensure model file is generated
!python models/train_model.py

# 2. Reload module cache
if 'gateway.app' in sys.modules:
    importlib.reload(sys.modules['gateway.app'])

from fastapi.testclient import TestClient
from gateway.app import app

# 3. Fetch sample customer ID
conn = sqlite3.connect("infrastructure/enterprise_data.db")
cursor = conn.cursor()
cursor.execute("SELECT golden_customer_id FROM dim_customer_golden LIMIT 1")
sample_id = cursor.fetchone()[0]
conn.close()

# 4. Use TestClient as a context manager to trigger startup events!
with TestClient(app) as client:
    print(f"\n🔍 Testing live ML prediction for Customer ID: {sample_id[:12]}...")
    response = client.get(f"/predict/{sample_id}")
    print("🔥 PREDICTION STATUS CODE:", response.status_code)
    print("✨ PREDICTION RESPONSE JSON:", response.json())

📊 Loaded 1000 Golden Records for Feature Engineering.

🎯 --- Model Training & Evaluation Metrics ---
  * Accuracy : 0.9850
  * Precision: 0.9850
  * Recall   : 1.0000
  * F1-Score : 0.9924

💾 Serialized Model saved successfully to '/content/enterprise-data-platform/models/customer_model.joblib'
🚀 [Startup] Loaded Machine Learning Model Artifact from '/content/enterprise-data-platform/models/customer_model.joblib'

🔍 Testing live ML prediction for Customer ID: 35508333febf...
🔥 PREDICTION STATUS CODE: 200
✨ PREDICTION RESPONSE JSON: {'golden_customer_id': '35508333febf1ddc32a8941d9a694274ec99dee44e334236f49ac2a2c9cf5607', 'engagement_prediction': 1, 'confidence_score': 0.9823, 'decision_status': 'HIGH_VALUE_TIER'}
